In [1]:
import os
data_path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/zeidler/too_big_for_git/features/'
path_to_test_func = os.path.join(data_path, 'filtered_features_balanced_Functional_test.json')
path_to_valid_func = os.path.join(data_path, 'filtered_features_balanced_Functional_valid.json')
path_to_test_lld = os.path.join(data_path, 'filtered_features_balanced_LLD_test.json')
path_to_valid_lld = os.path.join(data_path, 'filtered_features_balanced_LLD_valid.json')


In [11]:
import pandas as pd
df_test_func = pd.read_json(path_to_test_func, orient='index')
df_valid_func = pd.read_json(path_to_valid_func, orient='index')
df_test_lld = pd.read_json(path_to_test_lld, orient='index')
df_valid_lld = pd.read_json(path_to_valid_lld, orient='index')

In [12]:
df_original_func = pd.read_json(os.path.join(data_path, '../preprocess/ALC_features_Functional.json'), orient='index')
df_original_LLD = pd.read_json(os.path.join(data_path, '../preprocess/ALC_features_LLD.json'), orient='index')


In [7]:
def preprocess_index(s):
    last_two_parts_of_the_path = '/'.join(s.split('/')[-2:])
    last_two_parts_of_the_path_without_file_extension = last_two_parts_of_the_path.split('.')[0]
    common_path = last_two_parts_of_the_path_without_file_extension.strip('_annot')
    return common_path

meta_data_path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/zeidler/intoxicat/data/meta_data_annotation_all_features_130623.json'

meta_data_df = pd.read_json(meta_data_path, orient='index')
meta_data_df['common_path'] = meta_data_df.index.map(preprocess_index)

In [101]:
def save_df(filtered_df, out_filename):
    import json
    with open(out_filename, 'w') as jsn:
        json.dump(filtered_df.to_dict(orient='index'), jsn)
    print(f'Filtered dataframe saved to: {out_filename}')
    
def create_balanced_subdf(features_df, 
                          meta_data_df, 
                          exclude=[],
                          on='type', 
                          values_to_keep=['R', 'L'],
                          verbose=True,
                          save=True,
                          path_to_save='subsets/V2_functional_subset_type_RL.json',
                          same_n_per_on_per_speaker=True,
                          min_count=1
                         ):
    '''
    Subset R|L (as opposed to  E|M|D )
    '''
    #type_names = ['E', 'M', 'D']
    type_names = values_to_keep

    balanced_filtered_df = pd.DataFrame() #empty, new subframes for each speaker and type will be appended
    
    meta_data_df['common_path'] = meta_data_df.index.map(preprocess_index)
    features_df['common_path'] = features_df.index.map(preprocess_index)
    features_df = features_df.set_index('common_path')
    meta_data_df = meta_data_df.set_index('common_path')
    if verbose:
        print(f'Received features_df of length {len(features_df)}')
    
    for df in exclude:
        df['common_path'] = df.index.map(preprocess_index)
        df.set_index('common_path')
        features_df = features_df.loc[~features_df.index.isin(df.index)]
    if verbose:
        print(f'''Excluded from features_df data samples with undesired ids. 
        Remaining features_df is of length {len(features_df)}''')
    
    #now for each content group, select the same amount of samples for each speaker 
    #in intoxicated and not intoxicated
    #append after selecting
    all_speakers = meta_data_df['spn'].unique()
    if same_n_per_on_per_speaker:
        max_to_keep_per_speaker = min(features_df.join(meta_data_df[meta_data_df[on].isin(values_to_keep)], how='inner').groupby(['spn', 'alc', on]).size().reset_index(name='counts')['counts'])
        if max_to_keep_per_speaker < 1:
            print(f'Attention: max_to_keep_per_speaker={max_to_keep_per_speaker}')
            return
    
    for speaker in all_speakers:
        
        for content_type in type_names:
            #condition_spn_group = (meta_data_df['spn'] == speaker) & (meta_data_df['type'] == content_type) 
            condition_spn_group = (meta_data_df['spn'] == speaker) & (meta_data_df[on] == content_type) 
            
            condition_spn_group_a = condition_spn_group & (meta_data_df['alc'] == 'a') #intoxicated
            condition_spn_group_na = condition_spn_group & (meta_data_df['alc'] == 'na') #not intoxicated
 
            filtered_df_spn_group_a = meta_data_df[condition_spn_group_a].join(features_df, how='inner', lsuffix='_l')
            filtered_df_spn_group_na = meta_data_df[condition_spn_group_na].join(features_df, how='inner', lsuffix='_l')
            
            if verbose:
                print(f'''
                Found {len(filtered_df_spn_group_a)} intoxicated samples and
                {len(filtered_df_spn_group_na)} non-intoxicated ('na') samples for
                speaker='{speaker}', {on}='{content_type}'
                ''')
                
            #keep the same amount as in 'a' target
            #n_to_keep_per_type_and_speaker = m[content_type]
            if same_n_per_on_per_speaker:
                if len(filtered_df_spn_group_a) < min_count or len(filtered_df_spn_group_na) < min_count:
                    print(f'Speaker {speaker} has less than {min_count} data samples and will be excluded')
                    continue #disregard speaker
                    
                #for all speakers we keep the min per speaker per type
                filtered_df_spn_group_a = filtered_df_spn_group_a.sample(n=max_to_keep_per_speaker, random_state=42) #randomly keeps n rows for speaker and content group
                filtered_df_spn_group_na = filtered_df_spn_group_na.sample(n=max_to_keep_per_speaker, random_state=42) #randomly keeps n rows for speaker and content group
                
            
            else:
                #each speaker will have same amount of samples in 'a' and 'na', but different speakers may have different amount of datasamples
                if (len(filtered_df_spn_group_a) < len(filtered_df_spn_group_na)):
                    n_to_keep_per_type_and_speaker = len(filtered_df_spn_group_a)
                    filtered_df_spn_group_na = filtered_df_spn_group_na.sample(n=n_to_keep_per_type_and_speaker, random_state=42) #randomly keeps n rows for speaker and content group
                else:
                    n_to_keep_per_type_and_speaker = len(filtered_df_spn_group_na)
                    filtered_df_spn_group_a = filtered_df_spn_group_a.sample(n=n_to_keep_per_type_and_speaker, random_state=42) #randomly keeps n rows for speaker and content group

            #append
            balanced_filtered_df = pd.concat([balanced_filtered_df, filtered_df_spn_group_a, filtered_df_spn_group_na])

    #shuffle the rows        
    filtered_df = balanced_filtered_df.sample(frac=1, random_state=42) 
    print(f'{len(filtered_df)} samples kept overall')
    
    if verbose:
        print('Number of samples per speaker per group (should be a list with one number [n]):')
        print(filtered_df.groupby(['spn', 'alc']).size().reset_index(name='counts')['counts'].unique())
        if len(filtered_df.groupby(['spn', 'alc']).size().reset_index(name='counts')['counts'].unique()) > 1:
            print('''Attention: something may be wrong with the rebalancing. 
                  Different speakers have different amount of a and na samples''')
        print(filtered_df.groupby(['spn', 'alc']).size().reset_index(name='counts'))
    if save:
        if not path_to_save:
            path_to_save = f'subsets/XXX_{on}_{str(values_to_keep)}.json'
        save_df(filtered_df, path_to_save)
    return filtered_df


In [102]:
tmp_df = create_balanced_subdf(df_original_func, 
                               meta_data_df, 
                               exclude=[df_test_func, df_valid_func],
                               on='type', 
                               values_to_keep=['R', 'L'],
                               verbose=True,
                               save=False,
                               path_to_save='subsets/V2_functional_subset_type_RL.json')

Received features_df of length 15180
Excluded from features_df data samples with undesired ids. 
        Remaining features_df is of length 13236

                Found 10 intoxicated samples and
                27 non-intoxicated ('na') samples for
                speaker='544', type='R'
                

                Found 7 intoxicated samples and
                11 non-intoxicated ('na') samples for
                speaker='544', type='L'
                

                Found 10 intoxicated samples and
                24 non-intoxicated ('na') samples for
                speaker='50', type='R'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='50', type='L'
                

                Found 13 intoxicated samples and
                26 non-intoxicated ('na') samples for
                speaker='571', type='R'
                

                Found 6 intoxicated samples and
    


                Found 6 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='59', type='L'
                

                Found 10 intoxicated samples and
                23 non-intoxicated ('na') samples for
                speaker='25', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='25', type='L'
                

                Found 11 intoxicated samples and
                25 non-intoxicated ('na') samples for
                speaker='556', type='R'
                

                Found 7 intoxicated samples and
                11 non-intoxicated ('na') samples for
                speaker='556', type='L'
                

                Found 11 intoxicated samples and
                24 non-intoxicated ('na') samples for
                speaker='100', type='R'
                

                Found 6 intoxicated sam


                Found 4 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='501', type='L'
                

                Found 10 intoxicated samples and
                25 non-intoxicated ('na') samples for
                speaker='96', type='R'
                

                Found 6 intoxicated samples and
                11 non-intoxicated ('na') samples for
                speaker='96', type='L'
                

                Found 12 intoxicated samples and
                26 non-intoxicated ('na') samples for
                speaker='38', type='R'
                

                Found 6 intoxicated samples and
                11 non-intoxicated ('na') samples for
                speaker='38', type='L'
                

                Found 11 intoxicated samples and
                25 non-intoxicated ('na') samples for
                speaker='103', type='R'
                

                Found 4 intoxicated samp


                Found 3 intoxicated samples and
                11 non-intoxicated ('na') samples for
                speaker='591', type='L'
                

                Found 9 intoxicated samples and
                24 non-intoxicated ('na') samples for
                speaker='8', type='R'
                

                Found 5 intoxicated samples and
                11 non-intoxicated ('na') samples for
                speaker='8', type='L'
                

                Found 9 intoxicated samples and
                26 non-intoxicated ('na') samples for
                speaker='582', type='R'
                

                Found 7 intoxicated samples and
                11 non-intoxicated ('na') samples for
                speaker='582', type='L'
                

                Found 12 intoxicated samples and
                26 non-intoxicated ('na') samples for
                speaker='551', type='R'
                

                Found 5 intoxicated sample


                Found 10 intoxicated samples and
                27 non-intoxicated ('na') samples for
                speaker='521', type='R'
                

                Found 5 intoxicated samples and
                11 non-intoxicated ('na') samples for
                speaker='521', type='L'
                

                Found 11 intoxicated samples and
                25 non-intoxicated ('na') samples for
                speaker='27', type='R'
                

                Found 7 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='27', type='L'
                

                Found 13 intoxicated samples and
                24 non-intoxicated ('na') samples for
                speaker='79', type='R'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='79', type='L'
                

                Found 9 intoxicated sampl


                Found 11 intoxicated samples and
                22 non-intoxicated ('na') samples for
                speaker='62', type='R'
                

                Found 5 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='62', type='L'
                

                Found 12 intoxicated samples and
                25 non-intoxicated ('na') samples for
                speaker='576', type='R'
                

                Found 7 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='576', type='L'
                

                Found 13 intoxicated samples and
                26 non-intoxicated ('na') samples for
                speaker='88', type='R'
                

                Found 5 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='88', type='L'
                

                Found 11 intoxicated samp

In [103]:
tmp_df = create_balanced_subdf(df_original_func, 
                               meta_data_df, 
                               exclude=[],
                               on='type', 
                               values_to_keep=['R', 'L'],
                               verbose=True,
                               save=False,
                               path_to_save='subsets/V2_functional_subset_type_RL.json')

Received features_df of length 15180
Excluded from features_df data samples with undesired ids. 
        Remaining features_df is of length 15180

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='544', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='544', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='50', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='50', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='571', type='R'
                

                Found 7 intoxicated samples and
    


                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='25', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='556', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='556', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='100', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='100', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='58', type='R'
                

                Found 7 intoxicated sa


                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='96', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='38', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='38', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='103', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='103', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='46', type='R'
                

                Found 7 intoxicated samp


                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='550', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='591', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='591', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='8', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='8', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='582', type='R'
                

                Found 7 intoxicated samp


                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='536', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='536', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='6', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='6', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='64', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='64', type='L'
                

                Found 13 intoxicated sampl


                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='24', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='562', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='562', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='74', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='74', type='L'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='520', type='R'
                

                Found 7 intoxicated sam

In [104]:
tmp_df = create_balanced_subdf(df_original_func, 
                               meta_data_df, 
                               exclude=[],
                               on='type', 
                               values_to_keep=['R', 'L', 'E', 'M', 'D'],
                               verbose=True,
                               save=False,
                               path_to_save='subsets/V2_functional_subset_type_RL.json')

Received features_df of length 15180
Excluded from features_df data samples with undesired ids. 
        Remaining features_df is of length 15180

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='544', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='544', type='L'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='544', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='544', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='544', type='D'
                

                Found 13 intoxicated samples and
     


                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='508', type='L'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='508', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='508', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='508', type='D'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='95', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='95', type='L'
                

                Found 5 intoxicated sample


                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='101', type='L'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='101', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='101', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='101', type='D'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='505', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='505', type='L'
                

                Found 5 intoxicated samp


                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='502', type='L'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='502', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='502', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='502', type='D'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='507', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='507', type='L'
                

                Found 5 intoxicated samp


                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='15', type='D'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='596', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='596', type='L'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='596', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='596', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='596', type='D'
                

                Found 13 intoxicated sampl

...


                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='506', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='506', type='L'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='506', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='506', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='506', type='D'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='10', type='R'
                

                Found 7 intoxicated samp


                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='557', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='557', type='L'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='557', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='557', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='557', type='D'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='36', type='R'
                

                Found 7 intoxicated samp


                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='549', type='D'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='48', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='48', type='L'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='48', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='48', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='48', type='D'
                

                Found 13 intoxicated samples a


                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='576', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='576', type='L'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='576', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='576', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='576', type='D'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='88', type='R'
                

                Found 7 intoxicated samp


                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='90', type='L'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='90', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='90', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='90', type='D'
                

                Found 13 intoxicated samples and
                28 non-intoxicated ('na') samples for
                speaker='22', type='R'
                

                Found 7 intoxicated samples and
                12 non-intoxicated ('na') samples for
                speaker='22', type='L'
                

                Found 5 intoxicated samples an

In [105]:
tmp_df = create_balanced_subdf(df_original_func, 
                               meta_data_df, 
                               exclude=[],
                               on='type', 
                               values_to_keep=['E', 'M', 'D'],
                               verbose=True,
                               save=False,
                               path_to_save='subsets/V2_functional_subset_type_RL.json')

Received features_df of length 15180
Excluded from features_df data samples with undesired ids. 
        Remaining features_df is of length 15180

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='544', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='544', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='544', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='50', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='50', type='M'
                

                Found 2 intoxicated samples and
          


                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='552', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='552', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='57', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='57', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='57', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='9', type='E'
                

                Found 3 intoxicated samples and



                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='574', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='574', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='547', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='547', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='547', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='524', type='E'
                

                Found 3 intoxicated samples


                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='96', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='96', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='38', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='38', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='38', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='103', type='E'
                

                Found 3 intoxicated samples and



                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='534', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='534', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='23', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='23', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='23', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='92', type='E'
                

                Found 3 intoxicated samples and


                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='594', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='594', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='554', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='554', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='554', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='526', type='E'
                

                Found 3 intoxicated samples


                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='64', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='64', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='64', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='521', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='521', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='521', type='D'
                

                Found 5 intoxicated samples an


                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='48', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='48', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='53', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='53', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='53', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='500', type='E'
                

                Found 3 intoxicated samples and



                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='585', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='527', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='527', type='M'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='527', type='D'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='528', type='E'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='528', type='M'
                

                Found 2 intoxicated samples

In [106]:
path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/'
tmp_df = create_balanced_subdf(df_original_func, 
                               meta_data_df, 
                               exclude=[df_test_func, df_valid_func],
                               on='type', 
                               values_to_keep=['E', 'M', 'D', 'R', 'L'],
                               verbose=False,
                               save=True,
                               path_to_save=os.path.join(path, 'V2_bal_subset_functional_full.json'))
                    

Speaker 15 has less than 1 data samples and will be excluded
Speaker 543 has less than 1 data samples and will be excluded
Speaker 43 has less than 1 data samples and will be excluded
Speaker 23 has less than 1 data samples and will be excluded
Speaker 47 has less than 1 data samples and will be excluded
Speaker 82 has less than 1 data samples and will be excluded
Speaker 518 has less than 1 data samples and will be excluded
Speaker 88 has less than 1 data samples and will be excluded
1604 samples kept overall
Filtered dataframe saved to: /mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/V2_bal_subset_functional_full.json


In [107]:
path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/'
tmp_df = create_balanced_subdf(df_original_func, 
                               meta_data_df, 
                               exclude=[df_test_func, df_valid_func],
                               on='type', 
                               values_to_keep=['E', 'M', 'D'],
                               verbose=False,
                               save=True,
                               path_to_save=os.path.join(path, 'V2_bal_subset_functional_type_EMD.json'))
                    

Speaker 15 has less than 1 data samples and will be excluded
Speaker 543 has less than 1 data samples and will be excluded
Speaker 43 has less than 1 data samples and will be excluded
Speaker 23 has less than 1 data samples and will be excluded
Speaker 47 has less than 1 data samples and will be excluded
Speaker 82 has less than 1 data samples and will be excluded
Speaker 518 has less than 1 data samples and will be excluded
Speaker 88 has less than 1 data samples and will be excluded
956 samples kept overall
Filtered dataframe saved to: /mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/V2_bal_subset_functional_type_EMD.json


In [108]:
path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/'
tmp_df = create_balanced_subdf(df_original_func, 
                               meta_data_df, 
                               exclude=[df_test_func, df_valid_func],
                               on='type', 
                               values_to_keep=['R', 'L'],
                               verbose=False,
                               save=True,
                               path_to_save=os.path.join(path, 'V2_bal_subset_functional_type_RL.json'))
                    

1944 samples kept overall
Filtered dataframe saved to: /mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/V2_bal_subset_functional_type_RL.json


In [46]:
#sandbox
features_df = df_original_func
meta_data_df['common_path'] = meta_data_df.index.map(preprocess_index)
features_df['common_path'] = features_df.index.map(preprocess_index)
features_df = features_df.set_index('common_path')
meta_data_df = meta_data_df.set_index('common_path')
features_df.index[:2]

Index(['ses4038/5444038020_h_00', 'ses4038/5444038033_h_00'], dtype='object', name='common_path')

In [ ]:
min(features_df.join(meta_data_df, how='inner').groupby(['spn', 'alc', 'type']).size().reset_index(name='counts')['counts'])

In [109]:
path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/'
tmp_df = create_balanced_subdf(df_original_LLD, 
                               meta_data_df, 
                               exclude=[df_test_lld, df_valid_lld],
                               on='type', 
                               values_to_keep=['E', 'M', 'D', 'R', 'L'],
                               verbose=False,
                               save=True,
                               path_to_save=os.path.join(path, 'V2_bal_subset_LLD_full.json'))
                    

Speaker 55 has less than 1 data samples and will be excluded
Speaker 555 has less than 1 data samples and will be excluded
Speaker 551 has less than 1 data samples and will be excluded
Speaker 79 has less than 1 data samples and will be excluded
Speaker 91 has less than 1 data samples and will be excluded
Speaker 518 has less than 1 data samples and will be excluded
Speaker 74 has less than 1 data samples and will be excluded
Speaker 90 has less than 1 data samples and will be excluded
1604 samples kept overall
Filtered dataframe saved to: /mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/V2_bal_subset_LLD_full.json


In [110]:
path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/'
tmp_df = create_balanced_subdf(df_original_LLD, 
                               meta_data_df, 
                               exclude=[df_test_lld, df_valid_lld],
                               on='type', 
                               values_to_keep=['E', 'M', 'D'],
                               verbose=False,
                               save=True,
                               path_to_save=os.path.join(path, 'V2_bal_subset_LLD_type_EMD.json'))
                    

Speaker 55 has less than 1 data samples and will be excluded
Speaker 555 has less than 1 data samples and will be excluded
Speaker 551 has less than 1 data samples and will be excluded
Speaker 79 has less than 1 data samples and will be excluded
Speaker 91 has less than 1 data samples and will be excluded
Speaker 518 has less than 1 data samples and will be excluded
Speaker 74 has less than 1 data samples and will be excluded
Speaker 90 has less than 1 data samples and will be excluded
956 samples kept overall
Filtered dataframe saved to: /mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/V2_bal_subset_LLD_type_EMD.json


In [111]:
path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/'
tmp_df = create_balanced_subdf(df_original_LLD, 
                               meta_data_df, 
                               exclude=[df_test_lld, df_valid_lld],
                               on='type', 
                               values_to_keep=['R', 'L'],
                               verbose=False,
                               save=True,
                               path_to_save=os.path.join(path, 'V2_bal_subset_LLD_type_RL.json'))
                    

1296 samples kept overall
Filtered dataframe saved to: /mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/V2_bal_subset_LLD_type_RL.json


In [88]:
df_test_lld_ids = set(df_test_lld.index)
df_test_func_ids = set(df_test_func.index)
df_valid_lld_ids = set(df_valid_lld.index)
df_valid_func_ids = set(df_valid_func.index)

In [90]:
len(df_test_lld_ids - df_test_func_ids)

906

In [91]:
len(df_test_func_ids - df_test_lld_ids)

906

In [92]:
len(df_valid_lld_ids - df_valid_func_ids)

910

In [93]:
len(df_valid_func_ids - df_valid_lld_ids)

910

In [98]:
df_test_func.content.unique()

array(['P', 'R', 'N', 'T', 'A', 'Q', 'C', 'S'], dtype=object)

In [112]:
path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/'
tmp_df = create_balanced_subdf(df_original_LLD, 
                               meta_data_df, 
                               exclude=[df_test_lld, df_valid_lld],
                               on='content', 
                               values_to_keep=['P', 'R', 'N', 'T', 'A', 'Q', 'C', 'S'],
                               verbose=True,
                               save=True,
                               path_to_save=os.path.join(path, 'V2_bal_subset_LLD_content_full.json'))
                    

Received features_df of length 15180
Excluded from features_df data samples with undesired ids. 
        Remaining features_df is of length 13236

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='544', content='P'
                

                Found 4 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='544', content='R'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='544', content='N'
                

                Found 4 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='544', content='T'
                

                Found 5 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='544', content='A'
                

                Found 2 intoxicated sample


                Found 1 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='511', content='T'
                

                Found 4 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='511', content='A'
                

                Found 2 intoxicated samples and
                3 non-intoxicated ('na') samples for
                speaker='511', content='Q'
                

                Found 3 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='511', content='C'
                

                Found 1 intoxicated samples and
                1 non-intoxicated ('na') samples for
                speaker='511', content='S'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='11', content='P'
                

                Found 4 into


                Found 4 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='97', content='A'
                

                Found 1 intoxicated samples and
                4 non-intoxicated ('na') samples for
                speaker='97', content='Q'
                

                Found 3 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='97', content='C'
                

                Found 1 intoxicated samples and
                1 non-intoxicated ('na') samples for
                speaker='97', content='S'
                

                Found 2 intoxicated samples and
                6 non-intoxicated ('na') samples for
                speaker='49', content='P'
                

                Found 2 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='49', content='R'
                

                Found 3 intoxicat


                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='57', content='A'
                

                Found 1 intoxicated samples and
                3 non-intoxicated ('na') samples for
                speaker='57', content='Q'
                

                Found 4 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='57', content='C'
                

                Found 1 intoxicated samples and
                1 non-intoxicated ('na') samples for
                speaker='57', content='S'
                

                Found 3 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='9', content='P'
                

                Found 2 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='9', content='R'
                

                Found 4 intoxicat


                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='509', content='C'
                

                Found 0 intoxicated samples and
                1 non-intoxicated ('na') samples for
                speaker='509', content='S'
                
Speaker 509 has less than 1 data samples and will be excluded

                Found 3 intoxicated samples and
                6 non-intoxicated ('na') samples for
                speaker='16', content='P'
                

                Found 4 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='16', content='R'
                

                Found 1 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='16', content='N'
                

                Found 3 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='16'

...


                Found 4 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='48', content='R'
                

                Found 4 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='48', content='N'
                

                Found 4 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='48', content='T'
                

                Found 3 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='48', content='A'
                

                Found 1 intoxicated samples and
                4 non-intoxicated ('na') samples for
                speaker='48', content='Q'
                

                Found 4 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='48', content='C'
                

                Found 1 intoxic


                Found 2 intoxicated samples and
                4 non-intoxicated ('na') samples for
                speaker='24', content='Q'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='24', content='C'
                

                Found 1 intoxicated samples and
                1 non-intoxicated ('na') samples for
                speaker='24', content='S'
                

                Found 1 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='562', content='P'
                

                Found 4 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='562', content='R'
                

                Found 4 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='562', content='N'
                

                Found 3 intox


                Found 2 intoxicated samples and
                3 non-intoxicated ('na') samples for
                speaker='88', content='Q'
                

                Found 3 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='88', content='C'
                

                Found 1 intoxicated samples and
                1 non-intoxicated ('na') samples for
                speaker='88', content='S'
                

                Found 3 intoxicated samples and
                6 non-intoxicated ('na') samples for
                speaker='29', content='P'
                

                Found 4 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='29', content='R'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='29', content='N'
                

                Found 5 intoxica


                Found 1 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='528', content='P'
                

                Found 3 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='528', content='R'
                

                Found 2 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='528', content='N'
                

                Found 3 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='528', content='T'
                

                Found 3 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='528', content='A'
                

                Found 1 intoxicated samples and
                4 non-intoxicated ('na') samples for
                speaker='528', content='Q'
                

                Found 5 in

2498 samples kept overall
Number of samples per speaker per group (should be a list with one number [n]):
[8 7 6]
Attention: something may be wrong with the rebalancing. 
                  Different speakers have different amount of a and na samples
     spn alc  counts
0      6   a       8
1      6  na       8
2      7   a       7
3      7  na       7
4      8   a       8
..   ...  ..     ...
319  594  na       8
320  595   a       8
321  595  na       8
322  596   a       8
323  596  na       8

[324 rows x 3 columns]
Filtered dataframe saved to: /mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/V2_bal_subset_LLD_content_full.json


In [113]:
path = '/mount/arbeitsdaten/studenten1/team-lab-phonetics/2023/student_directories/kolos/CLTeamLab/too_big_for_git/subsets/'
tmp_df = create_balanced_subdf(df_original_func, 
                               meta_data_df, 
                               exclude=[df_test_func, df_valid_func],
                               on='content', 
                               values_to_keep=['P', 'R', 'N', 'T', 'A', 'Q', 'C', 'S'],
                               verbose=True,
                               save=True,
                               path_to_save=os.path.join(path, 'V2_bal_subset_functional_content_full.json'))
                    

Received features_df of length 15180
Excluded from features_df data samples with undesired ids. 
        Remaining features_df is of length 13236

                Found 3 intoxicated samples and
                6 non-intoxicated ('na') samples for
                speaker='544', content='P'
                

                Found 3 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='544', content='R'
                

                Found 5 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='544', content='N'
                

                Found 4 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='544', content='T'
                

                Found 4 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='544', content='A'
                

                Found 2 intoxicated sample


                Found 5 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='511', content='A'
                

                Found 2 intoxicated samples and
                4 non-intoxicated ('na') samples for
                speaker='511', content='Q'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='511', content='C'
                

                Found 1 intoxicated samples and
                1 non-intoxicated ('na') samples for
                speaker='511', content='S'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='11', content='P'
                

                Found 3 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='11', content='R'
                

                Found 5 into


                Found 3 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='97', content='T'
                

                Found 5 intoxicated samples and
                6 non-intoxicated ('na') samples for
                speaker='97', content='A'
                

                Found 0 intoxicated samples and
                4 non-intoxicated ('na') samples for
                speaker='97', content='Q'
                
Speaker 97 has less than 1 data samples and will be excluded

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='97', content='C'
                

                Found 1 intoxicated samples and
                1 non-intoxicated ('na') samples for
                speaker='97', content='S'
                

                Found 2 intoxicated samples and
                5 non-intoxicated ('na') samples for
                speaker='49', 


                Found 3 intoxicated samples and
                6 non-intoxicated ('na') samples for
                speaker='57', content='P'
                

                Found 2 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='57', content='R'
                

                Found 4 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='57', content='N'
                

                Found 3 intoxicated samples and
                7 non-intoxicated ('na') samples for
                speaker='57', content='T'
                

                Found 5 intoxicated samples and
                7 non-intoxicated ('na') samples for
                speaker='57', content='A'
                

                Found 2 intoxicated samples and
                4 non-intoxicated ('na') samples for
                speaker='57', content='Q'
                

                Found 3 intoxicat


                Found 3 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='509', content='C'
                

                Found 1 intoxicated samples and
                1 non-intoxicated ('na') samples for
                speaker='509', content='S'
                

                Found 3 intoxicated samples and
                3 non-intoxicated ('na') samples for
                speaker='16', content='P'
                

                Found 4 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='16', content='R'
                

                Found 5 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='16', content='N'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='16', content='T'
                

                Found 3 intoxi

...


                Found 2 intoxicated samples and
                6 non-intoxicated ('na') samples for
                speaker='73', content='P'
                

                Found 4 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='73', content='R'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='73', content='N'
                

                Found 3 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='73', content='T'
                

                Found 4 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='73', content='A'
                

                Found 2 intoxicated samples and
                4 non-intoxicated ('na') samples for
                speaker='73', content='Q'
                

                Found 4 intoxica


                Found 4 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='53', content='C'
                

                Found 1 intoxicated samples and
                1 non-intoxicated ('na') samples for
                speaker='53', content='S'
                

                Found 3 intoxicated samples and
                4 non-intoxicated ('na') samples for
                speaker='500', content='P'
                

                Found 4 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='500', content='R'
                

                Found 5 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='500', content='N'
                

                Found 5 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='500', content='T'
                

                Found 4 into


                Found 5 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='74', content='A'
                

                Found 2 intoxicated samples and
                4 non-intoxicated ('na') samples for
                speaker='74', content='Q'
                

                Found 3 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='74', content='C'
                

                Found 1 intoxicated samples and
                1 non-intoxicated ('na') samples for
                speaker='74', content='S'
                

                Found 2 intoxicated samples and
                6 non-intoxicated ('na') samples for
                speaker='520', content='P'
                

                Found 3 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='520', content='R'
                

                Found 3 intoxic


                Found 2 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='41', content='N'
                

                Found 5 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='41', content='T'
                

                Found 3 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='41', content='A'
                

                Found 1 intoxicated samples and
                4 non-intoxicated ('na') samples for
                speaker='41', content='Q'
                

                Found 4 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='41', content='C'
                

                Found 0 intoxicated samples and
                1 non-intoxicated ('na') samples for
                speaker='41', content='S'
                
Speaker 41 has less than 1 data s


                Found 2 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='545', content='C'
                

                Found 1 intoxicated samples and
                0 non-intoxicated ('na') samples for
                speaker='545', content='S'
                
Speaker 545 has less than 1 data samples and will be excluded

                Found 1 intoxicated samples and
                6 non-intoxicated ('na') samples for
                speaker='94', content='P'
                

                Found 3 intoxicated samples and
                8 non-intoxicated ('na') samples for
                speaker='94', content='R'
                

                Found 4 intoxicated samples and
                9 non-intoxicated ('na') samples for
                speaker='94', content='N'
                

                Found 4 intoxicated samples and
                10 non-intoxicated ('na') samples for
                speaker='94